# GwenLand glcuda Wave 135 - two-accumulator decode GEMV

Production A-B-B-A for per-row Q8 SoA decode GEMV instruction-level parallelism.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave135-gemv-ilp2-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3e2622b7137f014f11b6892000b5c7be74b67c5f"
SOURCE_REV = "7ad8ba686965ed609013f210f94da9fec5ecd006"
PATCH_SHA256 = "42d5354b477e184b5f558b21766e921e28fdd96ce8c4fecd46906d7b5ff83948"
PATCH_GZIP_B64 = """..."""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/tmp/wave135")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave135-gemv-ilp2-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave135.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}
    candidate_switch_env = {
        **common,
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_DECODE_Q8_GLUE": "1",
        "GLCUDA_DECODE_GEMV_ILP2": "1",
    }

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "76 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if (parity.returncode or "37 passed" not in parity.stdout
        or "0 failed" not in parity.stdout):
        raise RuntimeError("CUDA parity failed")

    phase = "graph-replay"
    graph = run([cargo, "test", "--release", "-p", "glcuda", "--test", "graph_replay",
                 "--locked", "--", "--nocapture", "--test-threads=1"],
                cwd=TREE, env=candidate_switch_env, check=False)
    save("cargo-graph-replay.log", graph)
    if graph.returncode or "4 passed" not in graph.stdout or "0 failed" not in graph.stdout:
        raise RuntimeError("Wave 132 graph replay gate failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([
        ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda.ptx",
        "-o", ROOT / "wave135.cubin",
    ], check=False)
    save("ptxas-wave135.log", resource)
    resource_log = resource.stdout + "\n" + resource.stderr
    candidate_section = re.search(
        r"Function properties for gl_gemv_q8_0_soa_ilp2(.*?)"
        r"(?=ptxas info    : Compiling entry function|\Z)",
        resource_log,
        re.S,
    )
    if candidate_section is None:
        raise RuntimeError("Wave 135 kernel missing from ptxas output")
    candidate_resource_log = candidate_section.group(1)
    required_zero = [
        "0 bytes stack frame", "0 bytes spill stores", "0 bytes spill loads",
    ]
    if resource.returncode or any(item not in candidate_resource_log for item in required_zero):
        raise RuntimeError(f"Wave 135 resource gate failed: {candidate_resource_log[-3000:]}")
    register_match = re.search(r"Used (\d+) registers", candidate_resource_log)
    if register_match is None:
        raise RuntimeError("Wave 135 register count missing")
    candidate_resources = {
        "registers": int(register_match.group(1)),
        "stack_bytes": 0,
        "spill_store_bytes": 0,
        "spill_load_bytes": 0,
        "threads": 256,
        "rows_per_block": 8,
    }

    retained_resource = run([
        ptxas, "-v", "-arch=sm_75",
        TREE / "glcuda/src/kernels/glcuda_sm75_wave129.ptx",
        "-o", ROOT / "wave131.cubin",
    ], check=False)
    save("ptxas-wave129.log", retained_resource)
    retained_resource_log = retained_resource.stdout + "\n" + retained_resource.stderr
    expected_retained = [
        "Used 80 registers", "used 1 barriers", "16384 bytes smem",
        "0 bytes stack frame", "0 bytes spill stores", "0 bytes spill loads",
    ]
    if (retained_resource.returncode
            or any(item not in retained_resource_log for item in expected_retained)):
        raise RuntimeError("Wave 131 retained resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "glbench-ceiling"
    glbench_build = run([cargo, "build", "--release", "-p", "glbench", "--locked"],
                        cwd=TREE, env=common)
    save("glbench-build.log", glbench_build)
    glbench = TARGET / "release/glbench"
    ceiling_prompt = ("Measure this deterministic systems prompt carefully. Explain how "
                      "token-parallel integer matrix multiplication uses shared memory, "
                      "Tensor Cores, and fixed launch geometry. " * 8)
    prod_env = {
        **common,
        "GLCUDA_FORCE_Q8": "1",
        "GLCUDA_GRID2D": "1",
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_Q8_NOSTORE": "1",
        "GLCUDA_FFN_GATE_UP_STACKED": "1",
        "GLCUDA_DEFER_FFN_RESIDUAL": "1",
        "GLCUDA_NTILE128": "1",
        "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1",
        "GLCUDA_GEMM_N16_PREFETCH": "1",
        "GLCUDA_N16_FUSED_SWIGLU": "1",
        "GLCUDA_ATTN_MMA4": "1",
        "GLCUDA_ATTN_MMA4_REGQ": "1",
        "GLCUDA_ATTN_MMA4_AV": "1",
        "GLCUDA_DECODE_Q8_GLUE": "1",
    }
    candidate_env = {**prod_env, "GLCUDA_DECODE_GEMV_ILP2": "1"}
    runs = []
    samples = {
        "control": {"prefill": [], "decode": []},
        "candidate": {"prefill": [], "decode": []},
    }
    for index, arm in enumerate(["control", "candidate", "candidate", "control"]):
        out = RESULTS / f"glbench-{index + 1}-{arm}.json"
        env = candidate_env if arm == "candidate" else prod_env
        measured = run([
            glbench, "run", "--engine", "glcuda", "--model", MODEL,
            "--prompt", ceiling_prompt, "--tokens", "64", "--cold-iters", "1",
            "--warmup", "2", "--iters", "10", "--temperature", "0", "--seed", "42",
            "--kind", "end_to_end", "--verify-against", "glproc", "--out", out,
        ], cwd=TREE, env=env, check=False)
        save(f"glbench-{index + 1}-{arm}.log", measured)
        if measured.returncode or not out.is_file():
            raise RuntimeError(f"glbench Wave 135 run failed: {index + 1} {arm}")
        dispatch_log = measured.stdout + "\n" + measured.stderr
        if ('"path":"wave129-fused-swiglu-production"' not in dispatch_log
                or '"path":"wave132-exact-q8-glue"' not in dispatch_log):
            raise RuntimeError(f"retained production dispatch missing: {index + 1} {arm}")
        ilp2_enabled = '"decode_gemv_ilp2":true' in dispatch_log
        ilp2_message = "Wave 135 two-accumulator decode GEMV enabled" in dispatch_log
        if ilp2_enabled != (arm == "candidate") or ilp2_message != (arm == "candidate"):
            raise RuntimeError(f"Wave 135 dispatch mismatch: {index + 1} {arm}")
        session = json.loads(out.read_text(encoding="utf-8"))
        validation_text = json.dumps(session.get("validation", {}))
        if (not session.get("validation", {}).get("passed")
                or "50/50 tokens match oracle" not in validation_text):
            raise RuntimeError(f"oracle gate failed: {index + 1} {arm}")
        for iteration in session["measurements"]["iterations"]:
            samples[arm]["prefill"].append(
                iteration["prompt_tokens"] * 1000.0 / iteration["prefill_ms"]
            )
            samples[arm]["decode"].append(
                iteration["generated_tokens"] * 1000.0 / iteration["decode_ms"]
            )
        runs.append({
            "order": index + 1,
            "arm": arm,
            "prefill_tps": session["analysis"]["prefill_tps"],
            "decode_tps": session["analysis"]["decode_tps"],
        })

    def distribution(values):
        return {
            "count": len(values),
            "mean": statistics.mean(values),
            "median": statistics.median(values),
            "p90": percentile(values, 0.90),
            "p99": percentile(values, 0.99),
            "min": min(values),
            "max": max(values),
            "median_ci95": bootstrap_ci(values, seed=135),
        }

    combined = {
        arm: {phase_name: distribution(values)
              for phase_name, values in phase_samples.items()}
        for arm, phase_samples in samples.items()
    }
    decode_delta_pct = 100.0 * (
        combined["candidate"]["decode"]["median"]
        / combined["control"]["decode"]["median"] - 1.0
    )
    prefill_delta_pct = 100.0 * (
        combined["candidate"]["prefill"]["median"]
        / combined["control"]["prefill"]["median"] - 1.0
    )
    candidate_session_medians = [
        run_record["decode_tps"]["median"] for run_record in runs
        if run_record["arm"] == "candidate"
    ]
    control_session_medians = [
        run_record["decode_tps"]["median"] for run_record in runs
        if run_record["arm"] == "control"
    ]
    directional = all(
        value > statistics.median(control_session_medians)
        for value in candidate_session_medians
    )
    retain = (
        decode_delta_pct > 0.5
        and directional
        and combined["candidate"]["prefill"]["median"] >= 15000.0
    )
    summary = {
        "wave": 135,
        "purpose": "two_accumulator_decode_gemv",
        "gpu": fields,
        "model": model_meta,
        "resources": candidate_resources,
        "runs": runs,
        "combined": combined,
        "decode_delta_pct": decode_delta_pct,
        "prefill_delta_pct": prefill_delta_pct,
        "candidate_sessions_directional": directional,
        "retained": retain,
        "decision": "retain" if retain else "reject",
    }
    report = [
        "# Wave 135 two-accumulator decode GEMV",
        "",
        f"- Decision: **{summary['decision']}**",
        f"- Decode P50: {combined['control']['decode']['median']:.3f} -> "
        f"{combined['candidate']['decode']['median']:.3f} tok/s "
        f"({decode_delta_pct:+.3f}%)",
        f"- Prefill P50: {combined['control']['prefill']['median']:.3f} -> "
        f"{combined['candidate']['prefill']['median']:.3f} tok/s "
        f"({prefill_delta_pct:+.3f}%)",
        f"- Candidate sessions directional: {directional}",
        f"- Registers/thread: {candidate_resources['registers']}; spills: 0 bytes",
        "- Oracle: 50/50 tokens in all four sessions",
    ]
    (RESULTS / "wave135-summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    (RESULTS / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("WAVE135_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
